In [91]:
from langgraph.graph import MessagesState
from pathlib import Path
from dotenv import load_dotenv
import json
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.types import Command
from langchain_community.retrievers import BM25Retriever
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document
from typing import Literal
from typing import Annotated
from langchain_core.messages import ToolMessage


load_dotenv(override=True)

True

In [92]:
class IncidentState(MessagesState):
    documentation_query: str
    retrieved_docs: list[Document]
    investigation_status: Literal[
    "INVESTIGATING",
    "PORTFOLIO_NOT_FOUND",
    "LOGS_NOT_FOUND",
    "NO_INCIDENT",
    "INCIDENT_FOUND"
]

In [93]:
#PORTFOLIO_PATH = Path(__file__).parent / "portfolio_executions.json"

PORTFOLIO_PATH = Path(r"C:/Ai Proyecto/LangGraph/data/portfolio_executions.json")
TXT_LOADER_PATH = Path(r"C:\Ai Proyecto\LangGraph\data\execution_logs.txt")
DOCUMENTS_PATH = Path(r"C:\Ai Proyecto\LangGraph\data\documentation")

In [94]:
def get_json():
    with open(PORTFOLIO_PATH, encoding="utf-8") as f:
        return json.load(f)

def get_documents():
    dir_loader = DirectoryLoader(
            path=str(DOCUMENTS_PATH),
            glob="*.md",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"}
        )
    return dir_loader.load()

def get_logs():
    with open(TXT_LOADER_PATH, 'r', encoding='utf-8') as archivo:
        contenido = archivo.read()
        return contenido.splitlines()

In [95]:

@tool
def get_portfolio_executions(
    portfolio_id: str,
    tool_call_id: Annotated[str, InjectedToolCallId]
):
    """Get recent executions for a portfolio."""

    doc = get_json()

    executions = [
        e for e in doc
        if e.get("portfolio_id") == portfolio_id
    ]

    if not executions:
        return Command(
            update={
                "investigation_status": "PORTFOLIO_NOT_FOUND",
                "messages": [
                    ToolMessage(
                        content=f"No executions found for portfolio {portfolio_id}.",
                        tool_call_id=tool_call_id
                    )
                ]
            }
        )

    return executions



@tool
def get_execution_logs(
    execution_id: str,
    tool_call_id: Annotated[str, InjectedToolCallId]
):
    """Get logs for a specific execution."""

    logs = get_logs()

    filtered_logs = []
    capturando = False

    for log in logs:

        if log == f"[{execution_id}]":
            capturando = True
            continue

        if capturando and log.startswith("[RUN-"):
            break

        if capturando:
            filtered_logs.append(log)

    if not filtered_logs:
        return Command(
            update={
                "investigation_status": "LOGS_NOT_FOUND",

                "messages": [
                    ToolMessage(
                        content="No logs found for this execution.",
                        tool_call_id=tool_call_id
                    )
                ]
            }
        )

    return filtered_logs

In [96]:
def documentation_validation(state: IncidentState):
    """Search technical documentation and known incidents."""
    messages = state["messages"]

    search_query_response = llm.invoke([
        (
            "system",
            """Based on the investigation history, generate a concise English
            search query to retrieve technical documentation relevant to the
            suspected root cause.
            
            Include only the important technical terms.
            Return only the search query."""
        ),
        *messages
    ])

    search_query = search_query_response.content
    docs = get_documents()

    splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = [
        ("##", "section")]
        )

    chunks = []

    for doc in docs:
        doc_chunks = splitter.split_text(doc.page_content)

        for chunk in doc_chunks:
            chunk.metadata["source"] = doc.metadata["source"]

        chunks.extend(doc_chunks)

    retriever = BM25Retriever.from_documents(
        documents=chunks,
        k=3
    )

    results = retriever.invoke(search_query)


    return {
        "documentation_query": search_query,
        "retrieved_docs": results
    }




In [97]:
tools = [
    get_portfolio_executions,
    get_execution_logs
]

In [98]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)

In [99]:
def agent_node(state: IncidentState):

    system_prompt = """
You are a technical incident investigation agent.

Use the available tools to investigate the portfolio and its executions.

When you have enough evidence and no more tools are required:

- If there is evidence of an incident, start your final response with:
  INCIDENT_FOUND

- If there is no evidence of an incident, start your final response with:
  NO_INCIDENT

Do not report an incident unless there is sufficient evidence.
"""

    response = llm_with_tools.invoke([
        ("system", system_prompt),
        *state["messages"]
    ])

    # Mientras pida tools seguimos investigando
    if response.tool_calls:
        return {
            "messages": [response],
            "investigation_status": "INVESTIGATING"
        }

    # Ya ha terminado de investigar
    content = response.content.strip()

    if content.startswith("NO_INCIDENT"):
        status = "NO_INCIDENT"

    elif content.startswith("INCIDENT_FOUND"):
        status = "INCIDENT_FOUND"

    else:
        status = "INVESTIGATING"

    return {
        "messages": [response],
        "investigation_status": status
    }

In [100]:
def route_agent(state: IncidentState):

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    if state["investigation_status"] == "NO_INCIDENT":
        return END

    if state["investigation_status"] == "INCIDENT_FOUND":
        return "documentation_validation"

    raise ValueError(
        f"Unexpected investigation status: {state['investigation_status']}"
    )


def route_tools(state: IncidentState):

    if state["investigation_status"] in [
        "PORTFOLIO_NOT_FOUND",
        "LOGS_NOT_FOUND"
    ]:
        return END

    return "agent"

In [101]:
def final_synthesis(state: IncidentState):

    messages = state["messages"]
    docs = state["retrieved_docs"]

    documentation = "\n\n".join(
        f"Source: {doc.metadata}\n{doc.page_content}"
        for doc in docs
    )

    prompt = f"""
    You are producing the FINAL technical incident investigation report.

    You have:
    1. The complete investigation history, including execution data and logs.
    2. Technical documentation retrieved to validate the investigation.

    Technical documentation:
    {documentation}

    Produce a complete final report.

    You MUST include exactly these four sections:

    ## Probable Root Cause
    State the most likely root cause based on the available evidence.

    ## Execution Evidence
    List the concrete evidence found during the investigation.
    Include execution IDs, instruments, dates, warnings, values or other
    relevant details when available.

    ## Documentation Validation
    Explain whether the retrieved technical documentation supports the
    suspected root cause.
    Reference the relevant incident or documentation section when available.

    ## Recommended Next Action
    Describe the next technical actions that should be taken.

    Rules:
    - Use only evidence present in the investigation or retrieved documentation.
    - Do not invent facts.
    - Clearly distinguish observed evidence from conclusions.
    - If the evidence is insufficient, explicitly say so.
    """

    response = llm.invoke([
        ("system", prompt),
        *messages
    ])

    return {
        "messages": [response]
    }

In [102]:
builder = StateGraph(IncidentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("documentation_validation", documentation_validation)
builder.add_node("final_synthesis", final_synthesis)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    route_agent
)

builder.add_conditional_edges(
    "tools",
    route_tools
)

builder.add_edge(
    "documentation_validation",
    "final_synthesis"
)

builder.add_edge(
    "final_synthesis",
    END
)

graph = builder.compile()

In [105]:
initial_state = {
    "messages": [
        (
            "user",
            """Investigate PORT-555 and determine whether there is any incident
            affecting its latest execution.

            Identify the probable root cause only if there is sufficient evidence."""
        )
    ],
    "documentation_query": "",
    "retrieved_docs": [],
    "investigation_status": "INVESTIGATING"
}

for event in graph.stream(
    initial_state,
    stream_mode="updates"
):
    print(event)
    print("-" * 100)

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 188, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5d5b340d02', 'id': 'chatcmpl-EVwKRIkwcbnMTB2LGcgy5J9iCwuTe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a110ac-4be9-70d1-adc7-e99937cb65fc-0', tool_calls=[{'name': 'get_portfolio_executions', 'args': {'portfolio_id': 'PORT-555'}, 'id': 'call_k3beNKGU4ap8VdwtRl6Hta8t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 21, '